# BERTimbau Sentiment Analysis PT-BR 🇧🇷

Fine-tuning do **BERTimbau Base** para classificação binária de sentimentos em avaliações de e-commerce em português brasileiro.

**Resultado final no teste:** Accuracy **96,92%** · F1 **97,81%** · Macro F1 **96,29%**

🤗 Modelo publicado: https://huggingface.co/caio16dev/bertimbau-sentiment-ptbr

Este notebook documenta o pipeline completo: preparação dos dados, prevenção de leakage, tokenização, fine-tuning, validação e avaliação final.

## 1. Ambiente e dependências

O experimento foi executado em ambiente com GPU. Para reproduzir em Kaggle/Colab, instale as dependências abaixo.

In [ ]:
!pip install -q transformers datasets accelerate evaluate huggingface_hub "pyarrow>=21,<24"

In [ ]:
import numpy as np
import pandas as pd
import torch

from datasets import Dataset, load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer

MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
PARQUET_URL = "https://huggingface.co/datasets/ruanchaves/b2w-reviews01/resolve/refs%2Fconvert%2Fparquet/default/train/0000.parquet"
RANDOM_STATE = 42

print("PyTorch:", torch.__version__)
print("GPU disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. Dataset

Foi utilizado o **B2W-Reviews01**, com 132.373 avaliações de consumidores brasileiros.

A versão Parquet é carregada diretamente para evitar incompatibilidade do script legado do dataset com versões atuais da biblioteca `datasets`.

In [ ]:
dataset = load_dataset("parquet", data_files=PARQUET_URL, split="train")
df = dataset.to_pandas()

print("Quantidade de linhas:", len(df))
print("Quantidade de colunas:", len(df.columns))
display(df.head())

In [ ]:
rating_distribution = df["overall_rating"].value_counts().sort_index()
display(rating_distribution)

print("Reviews sem texto:", df["review_text"].isna().sum())
print("Reviews com texto vazio:", (df["review_text"].fillna("").str.strip() == "").sum())
print("Reviews duplicadas:", df["review_text"].duplicated().sum())

## 3. Preparação do target e prevenção de leakage

O problema foi formulado como classificação binária:

- 1–2 estrelas → **NEGATIVO (0)**
- 3 estrelas → removidas por representarem uma região ambígua
- 4–5 estrelas → **POSITIVO (1)**

Textos vazios foram removidos. Também foram identificados textos duplicados e casos em que o mesmo texto aparecia com labels conflitantes. Esses casos foram tratados antes do split para evitar leakage entre treino e avaliação.

In [ ]:
df_model = df[df["overall_rating"].isin([1, 2, 4, 5])].copy()
df_model = df_model.dropna(subset=["review_text"])
df_model = df_model[df_model["review_text"].str.strip() != ""]
df_model["label"] = (df_model["overall_rating"] >= 4).astype(int)

label_por_texto = df_model.groupby("review_text")["label"].nunique()
textos_conflitantes = label_por_texto[label_por_texto > 1].index

df_clean = df_model[~df_model["review_text"].isin(textos_conflitantes)].copy()
df_clean = df_clean.drop_duplicates(subset=["review_text"])
df_clean = df_clean[["review_text", "label"]].reset_index(drop=True)

print("Quantidade final:", len(df_clean))
print(df_clean["label"].value_counts())
print(df_clean["label"].value_counts(normalize=True).mul(100))

## 4. Split estratificado

Foi usado split **80/10/10**, preservando a proporção das classes:

- treino: 88.704
- validação: 11.088
- teste: 11.089

O conjunto de teste foi mantido separado durante o desenvolvimento e usado apenas para a avaliação final.

In [ ]:
train_df, temp_df = train_test_split(df_clean, test_size=0.20, random_state=RANDOM_STATE, stratify=df_clean["label"])
valid_df, test_df = train_test_split(temp_df, test_size=0.50, random_state=RANDOM_STATE, stratify=temp_df["label"])

print("Treino:", train_df.shape)
print("Validação:", valid_df.shape)
print("Teste:", test_df.shape)

print("Positivos no treino:", train_df["label"].mean())
print("Positivos na validação:", valid_df["label"].mean())
print("Positivos no teste:", test_df["label"].mean())

## 5. Tokenização e escolha de `max_length`

O tokenizer oficial do BERTimbau foi utilizado.

A análise de 10.000 reviews mostrou mediana de 26 tokens, P95 de 97 e P99 de 179. Foi escolhido `max_length=128` como compromisso entre cobertura do texto e eficiência computacional.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

texto_exemplo = "O produto chegou rápido e funciona muito bem."
print(tokenizer.tokenize(texto_exemplo))
print(tokenizer(texto_exemplo, truncation=True, max_length=128))

In [ ]:
sample_texts = df_clean["review_text"].sample(n=min(10000, len(df_clean)), random_state=RANDOM_STATE).tolist()
token_lengths = [len(tokenizer.encode(text, add_special_tokens=True, truncation=False)) for text in sample_texts]

print("Mediana:", np.median(token_lengths))
print("Percentil 90:", np.percentile(token_lengths, 90))
print("Percentil 95:", np.percentile(token_lengths, 95))
print("Percentil 99:", np.percentile(token_lengths, 99))
print("Máximo:", np.max(token_lengths))
print(f"Cobertura até 128 tokens: {np.mean(np.array(token_lengths) <= 128) * 100:.2f}%")

In [ ]:
train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
valid_dataset = Dataset.from_pandas(valid_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

def tokenize_function(batch):
    return tokenizer(batch["review_text"], truncation=True, max_length=128)

train_tokenized = train_dataset.map(tokenize_function, batched=True)
valid_tokenized = valid_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

## 6. Modelo e fine-tuning

Foi utilizado `BertForSequenceClassification` com duas classes. O BERTimbau já possui pesos pré-treinados em português; a cabeça de classificação é criada para a tarefa downstream e ajustada junto ao modelo durante o fine-tuning.

In [ ]:
id2label = {0: "NEGATIVO", 1: "POSITIVO"}
label2id = {"NEGATIVO": 0, "POSITIVO": 1}

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average="binary", zero_division=0)
    _, _, f1_macro, _ = precision_recall_fscore_support(labels, predictions, average="macro", zero_division=0)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f1_macro": f1_macro
    }

In [ ]:
training_args = TrainingArguments(
    output_dir="./bertimbau-sentiment-ptbr",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=2,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=200,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=RANDOM_STATE
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=valid_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

train_result = trainer.train()

## 7. Resultados

### Validação

| Métrica | Resultado |
|---|---:|
| Accuracy | 97,25% |
| Precision | 98,22% |
| Recall | 97,88% |
| F1 | 98,05% |
| Macro F1 | 96,69% |

### Teste

| Métrica | Resultado |
|---|---:|
| Accuracy | **96,92%** |
| Precision | **97,95%** |
| Recall | **97,68%** |
| F1 | **97,81%** |
| Macro F1 | **96,29%** |

A diferença pequena entre validação e teste indica boa generalização do modelo.

In [ ]:
validation_results = trainer.evaluate()

print("Resultados de validação:")
for metric, value in validation_results.items():
    print(f"{metric}: {value}")

test_output = trainer.predict(test_tokenized)

print("\nResultados no conjunto de teste:")
for metric, value in test_output.metrics.items():
    print(f"{metric}: {value}")

In [ ]:
test_predictions = np.argmax(test_output.predictions, axis=1)

print(classification_report(
    test_output.label_ids,
    test_predictions,
    target_names=["NEGATIVO", "POSITIVO"],
    digits=4
))

cm = confusion_matrix(test_output.label_ids, test_predictions)
print("Matriz de confusão:")
print(cm)

## 8. Inferência

O modelo foi testado também com frases fora do dataset para verificar seu comportamento em exemplos simples e em uma frase com sentimento misto.

In [ ]:
frases_teste = [
    "Produto excelente, chegou rápido e funciona perfeitamente.",
    "Péssima compra, o produto chegou quebrado e ninguém resolveu meu problema.",
    "O produto é bom, mas a entrega demorou mais do que eu esperava.",
    "Não gostei. Qualidade muito abaixo do esperado.",
    "Gostei muito e compraria novamente."
]

model.eval()
inputs = tokenizer(frases_teste, padding=True, truncation=True, max_length=128, return_tensors="pt")
inputs = {key: value.to(model.device) for key, value in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)

probabilities = torch.softmax(outputs.logits, dim=1)
predictions = torch.argmax(probabilities, dim=1)

for frase, pred, probs in zip(frases_teste, predictions, probabilities):
    classe = id2label[pred.item()]
    confianca = probs[pred.item()].item()
    print(f"\nTexto: {frase}")
    print(f"Sentimento: {classe}")
    print(f"Confiança: {confianca:.2%}")

## 9. Modelo publicado

O modelo final está disponível publicamente no Hugging Face:

**https://huggingface.co/caio16dev/bertimbau-sentiment-ptbr**

Exemplo de uso:

```python
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="caio16dev/bertimbau-sentiment-ptbr"
)

print(classifier("Produto excelente, chegou rápido e funciona perfeitamente."))
```

### Limitações

- O domínio de treino é e-commerce.
- O problema é binário e não possui classe neutra.
- Reviews de 3 estrelas foram removidos.
- Os scores do classificador não passaram por calibração específica.